# 🏋️ BioFitCoach — Exercise Classification Model Training

**Model:** Bidirectional LSTM (BiLSTM)  
**Dataset:** [Kaggle — Workout/Fitness Video Dataset](https://www.kaggle.com/datasets/hasyimabdillah/workoutfitness-video)  
**Target Accuracy:** 95–99% (based on published research using this exact dataset)

---

## 📋 Notebook Flow

```
PHASE 1 — Setup & Dataset Download
PHASE 2 — Feature Extraction  (MediaPipe → landmarks + angles → CSV)
PHASE 3 — Data Preparation    (sequences, balancing, scaling, splitting)
PHASE 4 — Model Training      (BiLSTM — proven to achieve 99% on this dataset)
PHASE 5 — Evaluation          (confusion matrix, classification report)
PHASE 6 — Save Artifacts      (model.h5, scaler.pkl, label_mapping.json)
```

## 🔑 Key Vocabulary
- **BiLSTM**: Bidirectional LSTM — reads the sequence both forward AND backward, capturing richer temporal patterns than a standard GRU
- **Landmark**: A specific body joint detected by MediaPipe (e.g. left knee = index 25)
- **Sequence**: A fixed-length window of consecutive frames fed to the BiLSTM as one training sample
- **StandardScaler**: Transforms features to zero mean and unit variance — essential for neural network training
- **Label Encoding**: Converting string class names ('squat') to integers (0, 1, 2...)

---
# PHASE 1 — Setup & Installation

In [ ]:
# ── Install required packages ─────────────────────────────────────────────
# Run this cell once. Restart kernel after installation if needed.
import subprocess, sys

packages = [
    'mediapipe',
    'opencv-python',
    'tensorflow',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'tqdm',
    'kaggle',           # to download dataset programmatically
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed')

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────
import os
import cv2
import json
import math
import joblib
import warnings
import numpy as np
import pandas as pd
import mediapipe as mp
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# ML
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Deep learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

print(f'TensorFlow version : {tf.__version__}')
print(f'MediaPipe version  : {mp.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
# ⚠️  SET THESE PATHS BEFORE RUNNING

# Root folder of the Kaggle dataset after downloading/extracting
# Expected structure inside:
#   workoutfitness-video/
#       Barbell Biceps Curl/
#           video1.mp4 ...
#       Squat/
#           video1.mp4 ...
#       ...
DATASET_ROOT = './data/workoutfitness-video'   # ← change if needed

# Where to save extracted features CSV
FEATURES_CSV = './data/features_extracted.csv'

# Where to save model artifacts (matches your BioFitCoach project paths)
ARTIFACTS_DIR = './ml_models'
MODEL_PATH    = f'{ARTIFACTS_DIR}/gru_model.h5'           # name kept for API compatibility
SCALER_PATH   = f'{ARTIFACTS_DIR}/scaler.pkl'
LABEL_MAP_PATH= f'{ARTIFACTS_DIR}/label_mapping.json'

# ── Model hyperparameters ─────────────────────────────────────────────────
SEQUENCE_LENGTH   = 30      # frames per training sample (30 works best per research)
STRIDE            = 5       # step between windows (smaller = more samples)
BATCH_SIZE        = 64
MAX_EPOCHS        = 100
LEARNING_RATE     = 1e-3
CONFIDENCE_THRESH = 0.70    # must match shared/config.py

# ── Exercise classes — ALL 22 from the Kaggle dataset ────────────────────
# Confirmed folder names from: hasyimabdillah/workoutfitness-video
# The dataset uses LOWERCASE folder names with spaces.
# Both lowercase and Title Case variants are included to handle
# any capitalisation differences across OS/download tools.
#
# ⚠️  IMPORTANT: When you download the dataset, run the inspection
#     cell below FIRST to see your exact folder names, then
#     confirm they match one of the keys listed here.
#
# 🏋️  Exercise categories:
#   ARMS    : barbell_biceps_curl, hammer_curl, tricep_dips, tricep_pushdown
#   CHEST   : bench_press, decline_bench_press, incline_bench_press, chest_fly_machine, pushup
#   BACK    : deadlift, romanian_deadlift, lat_pulldown, t_bar_row, pull_up
#   LEGS    : squat, leg_extension, leg_raises, hip_thrust
#   SHOULDERS: lateral_raise, shoulder_press
#   CORE    : plank, russian_twist
EXERCISE_MAP = {
    # ── Lowercase (actual Kaggle folder names) ────────────────────────────
    'barbell biceps curl':  'barbell_biceps_curl',
    'bench press':          'bench_press',
    'chest fly machine':    'chest_fly_machine',
    'deadlift':             'deadlift',
    'decline bench press':  'decline_bench_press',
    'hammer curl':          'hammer_curl',
    'hip thrust':           'hip_thrust',
    'incline bench press':  'incline_bench_press',
    'lat pulldown':         'lat_pulldown',
    'lateral raises':       'lateral_raise',
    'leg extension':        'leg_extension',
    'leg raises':           'leg_raises',
    'plank':                'plank',
    'pull up':              'pull_up',
    'push up':              'pushup',
    'romanian deadlift':    'romanian_deadlift',
    'russian twist':        'russian_twist',
    'shoulder press':       'shoulder_press',
    'squat':                'squat',
    't bar row':            't_bar_row',
    'tricep dips':          'tricep_dips',
    'tricep pushdown':      'tricep_pushdown',

    # ── Title Case variants (fallback if your OS capitalises folders) ─────
    'Barbell Biceps Curl':  'barbell_biceps_curl',
    'Bench Press':          'bench_press',
    'Chest Fly Machine':    'chest_fly_machine',
    'Deadlift':             'deadlift',
    'Decline Bench Press':  'decline_bench_press',
    'Hammer Curl':          'hammer_curl',
    'Hip Thrust':           'hip_thrust',
    'Incline Bench Press':  'incline_bench_press',
    'Lat Pulldown':         'lat_pulldown',
    'Lateral Raises':       'lateral_raise',
    'Leg Extension':        'leg_extension',
    'Leg Raises':           'leg_raises',
    'Plank':                'plank',
    'Pull Up':              'pull_up',
    'Push Up':              'pushup',
    'Romanian Deadlift':    'romanian_deadlift',
    'Russian Twist':        'russian_twist',
    'Shoulder Press':       'shoulder_press',
    'Squat':                'squat',
    'T Bar Row':            't_bar_row',
    'Tricep Dips':          'tricep_dips',
    'Tricep Pushdown':      'tricep_pushdown',
}

# Unique clean class names (what the model will output)
ALL_CLASSES = sorted(set(EXERCISE_MAP.values()))
print(f'Total unique exercise classes: {len(ALL_CLASSES)}')

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs('./data', exist_ok=True)
print('✅ Configuration loaded')
print(f'   Dataset root    : {DATASET_ROOT}')
print(f'   Artifacts dir   : {ARTIFACTS_DIR}')
print(f'   Sequence length : {SEQUENCE_LENGTH} frames')

---
# PHASE 1b — Dataset Download (Kaggle API)

**Option A — Kaggle API (automated):**  
1. Go to https://www.kaggle.com/settings → API → Create New Token  
2. Save `kaggle.json` to `~/.kaggle/kaggle.json`  
3. Run the cell below

**Option B — Manual download:**  
Download from https://www.kaggle.com/datasets/hasyimabdillah/workoutfitness-video  
Extract to `./data/workoutfitness-video/`  
Then skip to PHASE 2.

In [ ]:
# ── Option A: Download via Kaggle API ─────────────────────────────────────
# Skip this cell if you downloaded manually

import os
kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')

if os.path.exists(kaggle_json):
    os.system('kaggle datasets download -d hasyimabdillah/workoutfitness-video -p ./data --unzip')
    print('✅ Dataset downloaded and extracted to ./data/')
else:
    print('⚠️  kaggle.json not found.')
    print('   Either set up Kaggle API credentials, or download manually.')
    print('   Place videos in: ./data/workoutfitness-video/<ExerciseName>/<video>.mp4')

In [ ]:
# ── Inspect dataset folder structure ─────────────────────────────────────
dataset_path = Path(DATASET_ROOT)

if dataset_path.exists():
    print('📁 Dataset folders found:\n')
    video_exts = {'.mp4', '.avi', '.mov', '.mkv'}
    found_mapped, found_unmapped = [], []
    total_videos = 0
    for folder in sorted(dataset_path.iterdir()):
        if folder.is_dir():
            vids   = [f for f in folder.iterdir() if f.suffix.lower() in video_exts]
            mapped = EXERCISE_MAP.get(folder.name)
            if mapped:
                found_mapped.append(folder.name)
                total_videos += len(vids)
                print(f'   ✅ {folder.name:<30} → {mapped:<25} ({len(vids)} videos)')
            else:
                found_unmapped.append(folder.name)
                print(f'   ❌ {folder.name:<30} → NOT IN MAP (will be skipped)')
    print(f'\n   Mapped   : {len(found_mapped)}/22 classes')
    print(f'   Unmapped : {len(found_unmapped)} (skipped during extraction)')
    print(f'   Total videos to process: {total_videos}')
    if found_unmapped:
        print(f'\n   ⚠️  Unmapped folders: {found_unmapped}')
        print('   Add them to EXERCISE_MAP in the configuration cell if needed.')
else:
    print(f'❌ Dataset not found at: {DATASET_ROOT}')
    print('   Download the dataset first (see cell above).')

print(f'\n📋 All 22 expected clean class names:')
for i, cls in enumerate(ALL_CLASSES, 1):
    print(f'   {i:>2}. {cls}')

---
# PHASE 2 — Feature Extraction

## What features do we extract?

Based on the research paper that achieved 99% on this exact dataset, the best model uses **two feature groups combined**:

**Group 1 — Raw landmark coordinates (x, y, z)** for 13 key joints = 39 values  
These capture the absolute position of body parts. Different exercises have very different spatial configurations.

**Group 2 — Computed angles** = 9 values  
These capture the *shape* of the pose, independent of where the person is in the frame.

**Total per frame: 48 features**

> 📚 **Why both?** Raw coordinates tell the model WHERE the body is. Angles tell it HOW it's shaped. Together they let the model distinguish exercises that look similar in angles but differ in position (e.g. push-up vs plank).

In [ ]:
# ── MediaPipe landmark indices ────────────────────────────────────────────
# We use 13 key joints (ignoring face landmarks)
KEY_LANDMARKS = {
    'left_shoulder':  11, 'right_shoulder': 12,
    'left_elbow':     13, 'right_elbow':    14,
    'left_wrist':     15, 'right_wrist':    16,
    'left_hip':       23, 'right_hip':      24,
    'left_knee':      25, 'right_knee':     26,
    'left_ankle':     27, 'right_ankle':    28,
    'nose':            0,
}

def calculate_angle(a, b, c):
    """Angle in degrees at joint B between rays B→A and B→C."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cos_val = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_val, -1.0, 1.0))))

def extract_frame_features(landmarks):
    """
    Extract the 48-feature vector from one MediaPipe result.
    Returns None if any key landmark has low visibility.
    
    Features:
      - 13 joints × 3 coords (x,y,z) = 39 features  (Group 1)
      -  9 joint angles               =  9 features  (Group 2)
      Total = 48 features
    """
    lm = landmarks.landmark
    
    # Check visibility of all key landmarks
    for name, idx in KEY_LANDMARKS.items():
        if lm[idx].visibility < 0.5:
            return None
    
    # Helper to get [x, y] for angle calculation (normalized coords)
    def pt(name):
        idx = KEY_LANDMARKS[name]
        return [lm[idx].x, lm[idx].y]
    
    # ── Group 1: Raw x,y,z coordinates ───────────────────────────────────
    coords = []
    for name in KEY_LANDMARKS:  # preserves insertion order (Python 3.7+)
        idx = KEY_LANDMARKS[name]
        coords.extend([lm[idx].x, lm[idx].y, lm[idx].z])
    # coords has 13 × 3 = 39 values
    
    # ── Group 2: Computed joint angles ────────────────────────────────────
    angles = [
        calculate_angle(pt('left_shoulder'),  pt('left_elbow'),   pt('left_wrist')),     # left elbow
        calculate_angle(pt('right_shoulder'), pt('right_elbow'),  pt('right_wrist')),    # right elbow
        calculate_angle(pt('left_hip'),       pt('left_knee'),    pt('left_ankle')),     # left knee
        calculate_angle(pt('right_hip'),      pt('right_knee'),   pt('right_ankle')),    # right knee
        calculate_angle(pt('left_shoulder'),  pt('left_hip'),     pt('left_knee')),      # left hip
        calculate_angle(pt('right_shoulder'), pt('right_hip'),    pt('right_knee')),     # right hip
        calculate_angle(pt('left_elbow'),     pt('left_shoulder'),pt('right_shoulder')), # shoulder spread
        calculate_angle(pt('left_shoulder'),  pt('left_hip'),     pt('left_knee')),      # torso
        calculate_angle(pt('left_knee'),      pt('left_hip'),     pt('right_hip')),      # hip width
    ]
    # angles has 9 values
    
    return np.array(coords + angles, dtype=np.float32)   # shape: (48,)

NUM_FEATURES = 13 * 3 + 9   # = 48
print(f'✅ Feature extractor ready — {NUM_FEATURES} features per frame')
print(f'   Group 1 (raw coords): 13 joints × 3 = 39')
print(f'   Group 2 (angles):     9 joint angles')

In [ ]:
# ── Main extraction loop ──────────────────────────────────────────────────
# Processes all videos → saves CSV with one row per frame
# Expected runtime: 20–60 minutes depending on dataset size and CPU

mp_pose = mp.solutions.pose
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv'}
FRAME_SKIP = 2   # process every 2nd frame (balances speed vs data volume)

all_rows = []
dataset_path = Path(DATASET_ROOT)

print('🚀 Starting feature extraction...\n')

with mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    model_complexity=1,
) as pose:

    for folder in sorted(dataset_path.iterdir()):
        if not folder.is_dir():
            continue

        # Map folder name to our clean class name
        class_name = EXERCISE_MAP.get(folder.name)
        if class_name is None:
            print(f'  ⏭  Skipping unmapped folder: {folder.name}')
            continue

        video_files = [f for f in folder.iterdir() if f.suffix.lower() in VIDEO_EXTS]
        print(f'  [{class_name.upper():<15}] — {len(video_files)} videos')

        for video_path in tqdm(video_files, desc=f'  {class_name}', ncols=70):
            cap = cv2.VideoCapture(str(video_path))
            frame_num = 0
            video_id  = f'{class_name}_{video_path.stem}'

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                frame_num += 1
                if frame_num % FRAME_SKIP != 0:
                    continue

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                rgb.flags.writeable = False
                results = pose.process(rgb)

                if not results.pose_landmarks:
                    continue

                features = extract_frame_features(results.pose_landmarks)
                if features is None:
                    continue

                all_rows.append({
                    'video_id':  video_id,
                    'exercise':  class_name,
                    'frame_idx': frame_num,
                    **{f'f{i}': v for i, v in enumerate(features)}
                })

            cap.release()

df = pd.DataFrame(all_rows)
df.to_csv(FEATURES_CSV, index=False)

print(f'\n✅ Extraction complete!')
print(f'   Total frames : {len(df):,}')
print(f'   Saved to     : {FEATURES_CSV}')
print(f'\n   Class distribution:')
print(df['exercise'].value_counts().to_string())

---
# PHASE 3 — Data Preparation

Steps:
1. Load extracted CSV
2. Build sequences (sliding window of 30 frames per sample)
3. Balance classes (oversample minority exercises)
4. Train/Val/Test split (70/15/15)
5. StandardScaler (fit on train only — avoids data leakage)

> 📚 **Data Leakage**: If you fit the scaler on ALL data (including test), the model indirectly uses test statistics during training. Always fit the scaler ONLY on training data, then apply it to val/test.

In [ ]:
# ── Load and clean ────────────────────────────────────────────────────────
df = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in df.columns if c.startswith('f')]

# Drop rows with NaN or Inf (rare but possible)
before = len(df)
df = df.replace([np.inf, -np.inf], np.nan).dropna()
print(f'Dropped {before - len(df)} invalid rows')
print(f'Dataset: {len(df):,} frames | {df["exercise"].nunique()} classes | {len(feature_cols)} features')
print(f'\nClass counts:')
print(df['exercise'].value_counts())

In [ ]:
# ── Label encoding ────────────────────────────────────────────────────────
le = LabelEncoder()
df['label'] = le.fit_transform(df['exercise'])

label_mapping = {int(i): name for i, name in enumerate(le.classes_)}
NUM_CLASSES   = len(label_mapping)
CLASS_NAMES   = list(le.classes_)

print(f'Label mapping ({NUM_CLASSES} classes):')
for idx, name in label_mapping.items():
    print(f'   {idx} → {name}')

In [ ]:
# ── Build sequences (sliding window) ─────────────────────────────────────
# Each sample = 30 consecutive frames from ONE video
# Label = the exercise performed throughout those 30 frames

X_list, y_list = [], []

for video_id, group in tqdm(df.groupby('video_id'), desc='Building sequences'):
    frames = group[feature_cols].values    # shape: (num_frames, 48)
    label  = group['label'].iloc[0]
    
    # Slide window across this video's frames
    for start in range(0, len(frames) - SEQUENCE_LENGTH + 1, STRIDE):
        window = frames[start : start + SEQUENCE_LENGTH]
        X_list.append(window)
        y_list.append(label)

X = np.array(X_list, dtype=np.float32)   # (num_samples, 30, 48)
y = np.array(y_list, dtype=np.int32)      # (num_samples,)

print(f'X shape : {X.shape}  →  (samples, sequence_length, features)')
print(f'y shape : {y.shape}')
print(f'Total sequences : {len(X):,}')

In [ ]:
# ── Class balancing (oversampling with Gaussian noise) ───────────────────
# Ensures no exercise class dominates training

counts    = Counter(y)
max_count = max(counts.values())

X_parts, y_parts = [X], [y]

for cls, count in counts.items():
    shortage = max_count - count
    if shortage <= 0:
        continue
    idxs   = np.where(y == cls)[0]
    chosen = np.random.choice(idxs, size=shortage, replace=True)
    # Add tiny Gaussian noise (σ=0.01) to make synthetic copies slightly different
    # This prevents the model from memorising exact duplicates
    noise  = np.random.normal(0, 0.01, X[chosen].shape).astype(np.float32)
    X_parts.append(X[chosen] + noise)
    y_parts.append(y[chosen])

X_bal = np.concatenate(X_parts)
y_bal = np.concatenate(y_parts)

# Shuffle
perm  = np.random.permutation(len(X_bal))
X_bal, y_bal = X_bal[perm], y_bal[perm]

print(f'Before balancing : {len(X):,} samples')
print(f'After balancing  : {len(X_bal):,} samples')
print(f'Per-class counts (balanced): {Counter(y_bal.tolist())}')

In [ ]:
# ── Train / Val / Test split (70 / 15 / 15) ───────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X_bal, y_bal, test_size=0.15, stratify=y_bal, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/(1-0.15), stratify=y_temp, random_state=42
)

print(f'Train : {len(X_train):,}')
print(f'Val   : {len(X_val):,}')
print(f'Test  : {len(X_test):,}')

In [ ]:
# ── StandardScaler — fit ONLY on training data ────────────────────────────
scaler = StandardScaler()

# Reshape to 2D for scaler, then back to 3D
n_train, seq_len, n_feat = X_train.shape

X_train_s = scaler.fit_transform(X_train.reshape(-1, n_feat)).reshape(n_train, seq_len, n_feat)
X_val_s   = scaler.transform(X_val.reshape(-1, n_feat)).reshape(X_val.shape)
X_test_s  = scaler.transform(X_test.reshape(-1, n_feat)).reshape(X_test.shape)

print(f'✅ Scaler fitted on {n_train:,} training samples')
print(f'   X_train_s shape: {X_train_s.shape}')

---
# PHASE 4 — Model Training

## Architecture: Bidirectional LSTM

Based on the published research using this exact dataset, **BiLSTM consistently outperforms GRU** by 4–14 percentage points.

```
Input  (batch, 30, 48)
  ↓
Bidirectional LSTM(128)  ← reads sequence forward AND backward
  ↓
BatchNormalization + Dropout(0.4)
  ↓
Bidirectional LSTM(64)
  ↓
BatchNormalization + Dropout(0.3)
  ↓
Dense(128, relu) + Dropout(0.2)
  ↓
Dense(num_classes, softmax)
```

> 📚 **BiLSTM vs GRU**: An LSTM has an extra 'cell state' that helps it remember long-range dependencies (e.g. the start of a squat vs the end). Bidirectional means it processes the 30-frame window both forward and backward, so each frame has context from future frames too — which helps when classifying the middle of an exercise rep.

In [ ]:
# ── Build BiLSTM model ────────────────────────────────────────────────────
def build_bilstm(seq_len, n_features, n_classes):
    inputs = layers.Input(shape=(seq_len, n_features), name='pose_sequence')

    # First BiLSTM block
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
        name='bilstm_1'
    )(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    # Second BiLSTM block
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=False, dropout=0.2),
        name='bilstm_2'
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Classification head
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    model = models.Model(inputs, outputs, name='BiLSTM_ExerciseClassifier')
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model = build_bilstm(SEQUENCE_LENGTH, NUM_FEATURES, NUM_CLASSES)
model.summary()

In [ ]:
# ── Training callbacks ────────────────────────────────────────────────────
# EarlyStopping   : stops training when val_accuracy stops improving
# ModelCheckpoint : saves the BEST model (by val_accuracy)
# ReduceLROnPlateau: halves learning rate when val_loss stalls

callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1,
    ),
]

print('✅ Callbacks configured')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
# Expected time: 30–90 minutes on CPU | 5–15 minutes on GPU
# Early stopping will usually stop before MAX_EPOCHS

print(f'Training BiLSTM on {len(X_train_s):,} samples...')
print(f'Validation on {len(X_val_s):,} samples')
print(f'Max epochs: {MAX_EPOCHS} (early stopping will likely stop earlier)\n')

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

best_val_acc = max(history.history['val_accuracy'])
print(f'\n🏆 Best Val Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

---
# PHASE 5 — Evaluation

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history.history['accuracy']) + 1)

ax1.plot(epochs, history.history['accuracy'],     'b-o', markersize=3, label='Train')
ax1.plot(epochs, history.history['val_accuracy'], 'r-o', markersize=3, label='Validation')
ax1.set_title('Accuracy', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1.05])

ax2.plot(epochs, history.history['loss'],     'b-o', markersize=3, label='Train')
ax2.plot(epochs, history.history['val_loss'], 'r-o', markersize=3, label='Validation')
ax2.set_title('Loss', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('BiLSTM Training Curves — BioFitCoach', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────
# The test set was NEVER seen during training or validation
# This is the most honest measure of real-world performance

test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
y_pred_probs = model.predict(X_test_s, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print()
print('Per-class Classification Report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_pct,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    cbar_kws={'label': '% of True Class'},
)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title(f'Confusion Matrix — Test Accuracy: {test_acc*100:.2f}%', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/confusion_matrix.png', dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

---
# PHASE 6 — Save Artifacts

Three files are saved. All three are required at inference time:
- `gru_model.h5` — the trained BiLSTM model (name kept for API compatibility)
- `scaler.pkl` — the fitted StandardScaler
- `label_mapping.json` — int → exercise name mapping

Copy these to your BioFitCoach project `ml_models/` folder.

In [ ]:
# ── Save all three artifacts ──────────────────────────────────────────────
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# 1. Model (.h5 format — compatible with Keras load_model)
model.save(MODEL_PATH)
print(f'✅ Model saved      : {MODEL_PATH}')

# 2. Scaler (joblib pickle)
joblib.dump(scaler, SCALER_PATH)
print(f'✅ Scaler saved     : {SCALER_PATH}')

# 3. Label mapping (JSON — human readable)
with open(LABEL_MAP_PATH, 'w') as f:
    json.dump(label_mapping, f, indent=2)
print(f'✅ Label map saved  : {LABEL_MAP_PATH}')

# 4. Also save NUM_FEATURES and SEQUENCE_LENGTH for reference
config_summary = {
    'num_features':    NUM_FEATURES,
    'sequence_length': SEQUENCE_LENGTH,
    'num_classes':     NUM_CLASSES,
    'class_names':     CLASS_NAMES,
    'label_mapping':   label_mapping,
    'test_accuracy':   round(float(test_acc), 4),
    'best_val_accuracy': round(float(best_val_acc), 4),
}
with open(f'{ARTIFACTS_DIR}/model_config.json', 'w') as f:
    json.dump(config_summary, f, indent=2)
print(f'✅ Config saved     : {ARTIFACTS_DIR}/model_config.json')

print(f'\n📊 Final Results:')
print(f'   Best Val Accuracy  : {best_val_acc*100:.2f}%')
print(f'   Test Accuracy      : {test_acc*100:.2f}%')
print(f'   Classes trained    : {CLASS_NAMES}')
print(f'   Features per frame : {NUM_FEATURES}')
print(f'   Sequence length    : {SEQUENCE_LENGTH} frames')

print(f'''
📁 Next steps:
   1. Copy these files to your BioFitCoach project:
      {MODEL_PATH}    → FYP/ml_models/gru_model.h5
      {SCALER_PATH}   → FYP/ml_models/scaler.pkl
      {LABEL_MAP_PATH}→ FYP/ml_models/label_mapping.json
   
   2. Update shared/config.py:
      SEQUENCE_LENGTH = {SEQUENCE_LENGTH}  (change from 25 to 30)
      NUM_FEATURES    = {NUM_FEATURES}  (change from 21 to 48)
   
   3. Start the API: uvicorn main:app --reload
''')

---
# PHASE 7 — Update feature_engineering.py for Inference

⚠️ **Important**: Now that we use 48 features (not 21), we need to update the `extract_frame_features` function in `module_2_exercise_coach/services/feature_engineering.py` to match exactly what we trained on.

The cell below generates the exact code to paste into your `feature_engineering.py` file.

In [ ]:
# ── Print the exact feature_engineering.py update needed ─────────────────
print('''
📋 COPY THIS into feature_engineering.py (replace the existing constants):

NUM_FEATURES    = 48    # 13 joints × 3 coords + 9 angles
SEQUENCE_LENGTH = 30    # frames per GRU/BiLSTM input

KEY_LANDMARKS = {
    "left_shoulder":  11, "right_shoulder": 12,
    "left_elbow":     13, "right_elbow":    14,
    "left_wrist":     15, "right_wrist":    16,
    "left_hip":       23, "right_hip":      24,
    "left_knee":      25, "right_knee":     26,
    "left_ankle":     27, "right_ankle":    28,
    "nose":            0,
}

The extract_frame_features() function in this notebook
is the EXACT same function to use in feature_engineering.py.
Replace compute_features() with extract_frame_features().

Also update shared/config.py:
    sequence_length: int = 30   (was 25)
''')

In [ ]:
# ── Optional: Quick single-frame inference test ───────────────────────────
# Tests that the model, scaler, and label mapping all load correctly

print('🧪 Testing model reload...')

loaded_model  = tf.keras.models.load_model(MODEL_PATH)
loaded_scaler = joblib.load(SCALER_PATH)
with open(LABEL_MAP_PATH) as f:
    loaded_labels = json.load(f)

# Create a dummy sequence (zeros — just to test shapes)
dummy = np.zeros((1, SEQUENCE_LENGTH, NUM_FEATURES), dtype=np.float32)
dummy_scaled = loaded_scaler.transform(dummy.reshape(-1, NUM_FEATURES)).reshape(1, SEQUENCE_LENGTH, NUM_FEATURES)
probs = loaded_model.predict(dummy_scaled, verbose=0)

print(f'✅ Model reloaded successfully')
print(f'   Input shape  : {dummy_scaled.shape}')
print(f'   Output shape : {probs.shape}  (1 sample, {NUM_CLASSES} class probabilities)')
print(f'   Predicted class: {loaded_labels[str(np.argmax(probs[0]))]} ({np.max(probs[0]):.3f} confidence)')
print(f'\n🎉 All artifacts verified and ready for deployment!')